In [40]:
%load_ext autoreload
%autoreload 2
from experiment import find_all_datasets, find_all_experiments

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
dses = find_all_datasets("../../datasets/")
imagenet = dses["imagenet"]
split = "trainUval"

In [42]:
data_dir = "C:/home/eurovis_data/landscape_data_pt/"
strees_dir = "C:/home/eurovis_data/strees_pt/"

In [43]:
exps = find_all_experiments(dses, data_dir, strees_dir)
exp = exps[1]
assert exp.split == split

In [44]:
from basic_utils import get_labels, get_partition, get_tree, get_order_and_weights
labels = get_labels(exp)
partition = get_partition(exp)

assert len(partition) == len(labels)

In [45]:
import pyct as ct
import numpy as np
import pickle as pkl

data, _ = get_tree(exp)
simpl = ct.SimplifyCT() # type: ignore
simpl.setInput(data)

order, wts = get_order_and_weights(exp)

prop = 0.5

with open(f"results/result_{prop}.pkl", "rb") as f:
    fns, remaining_all, remaining_homo, maj_class_homo_cov, maj_class_homo_counts, class_homo_covs, class_all_covs = pkl.load(f)

In [46]:
import plotly.express as px

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(x=fns, y=remaining_homo, mode='lines', name=f'{prop*100:.1f}% Valleys', line=dict(color='blue', shape="hv")))
fig.add_trace(go.Scatter(x=fns, y=remaining_all, mode='lines', name='Valleys', line=dict(color='red', shape="hv")))

fig.show()

In [47]:
classes_needed = 525 # 1000 classes, the following criteria must apply to this many of them
maj_coverage_needed = 0.1 # at least this much coverage in homogeneous valleys where they are the majority class 
total_coverage_needed = 0.1 # at least this much total coverage in all valleys

# coverages are functions of accuracy, so this constraint necessarily tightens in more complex datasets

represented_counts = [[mc >= maj_coverage_needed and tc >= total_coverage_needed for mc, tc in zip(maj_cov, total_cov)].count(True) 
                      for maj_cov, total_cov in zip(maj_class_homo_cov, class_all_covs)]
represented = [count >= classes_needed for count in represented_counts]

# find interval where this is true
last_idx = len(fns) - represented[::-1].index(True) - 1

first_idx = represented.index(True)

print(f"Uniform Interval: {all(represented[first_idx:last_idx+1])}")
print(f"First IDX: {first_idx} - ")
print(first_idx, len(fns), fns[first_idx], list(zip(maj_class_homo_cov[first_idx], class_all_covs[first_idx], maj_class_homo_counts[first_idx])), sep="\n")
print(represented_counts[first_idx], remaining_all[first_idx], remaining_homo[first_idx])

print(f"\nLast IDX: {last_idx} - ")
print(last_idx, len(fns), fns[last_idx], list(zip(maj_class_homo_cov[last_idx], class_all_covs[last_idx], maj_class_homo_counts[last_idx])), sep="\n")
print(represented_counts[last_idx], remaining_all[last_idx], remaining_homo[last_idx])

Uniform Interval: False
First IDX: 3984 - 
3984
9942
0.00017019943334162235
[(0.4874074074074074, 0.4874074074074074, 1), (0.41703703703703704, 0.4177777777777778, 2), (0.05333333333333334, 0.05333333333333334, 4), (0.022962962962962963, 0.022962962962962963, 4), (0.017037037037037038, 0.017037037037037038, 7), (0.1451851851851852, 0.14592592592592593, 4), (0.02962962962962963, 0.02962962962962963, 22), (0.022962962962962963, 0.022962962962962963, 9), (0.027407407407407408, 0.027407407407407408, 21), (0.6755555555555556, 0.6755555555555556, 1), (0.11555555555555555, 0.11555555555555555, 2), (0.6955555555555556, 0.6955555555555556, 1), (0.5562962962962963, 0.5562962962962963, 1), (0.3451851851851852, 0.3459259259259259, 1), (0.7577777777777778, 0.7577777777777778, 1), (0.6844444444444444, 0.6844444444444444, 1), (0.6103703703703703, 0.6103703703703703, 1), (0.422962962962963, 0.422962962962963, 1), (0.4325925925925926, 0.4325925925925926, 1), (0.7755555555555556, 0.7755555555555556, 1),

In [48]:
from plotly.subplots import make_subplots
fig = make_subplots(1, 2, shared_yaxes=True, column_titles=["Class Homogeneous Coverage (start)", "Class Homogeneous Coverage (end)"])

fig.add_trace(go.Histogram(x=maj_class_homo_cov[first_idx], cumulative_enabled=True, xbins=dict(start=0, end=0.8, size=0.0032), marker_color="indianred", name="Start"), row=1, col=1)
fig.add_trace(go.Histogram(x=maj_class_homo_cov[last_idx], cumulative_enabled=True, xbins=dict(start=0, end=0.8, size=0.0032), marker_color="indianred", name="End"), row=1, col=2)

fig.update_layout(showlegend=False)

fig.show()

In [49]:
from plotly.subplots import make_subplots
fig = make_subplots(1, 2, shared_yaxes=True, column_titles=["Class Valley Coverage (start)", "Class Valley Coverage (end)"])

fig.add_trace(go.Histogram(x=class_all_covs[first_idx], cumulative_enabled=True, xbins=dict(start=0, end=0.8, size=0.0032), marker_color="indianred", name="Start"), row=1, col=1)
fig.add_trace(go.Histogram(x=class_homo_covs[last_idx], cumulative_enabled=True, xbins=dict(start=0, end=0.8, size=0.0032), marker_color="indianred", name="End"), row=1, col=2)

fig.update_layout(showlegend=False)

fig.show()